# LLMs as EHR encoders in PyHealth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/llm_ehr_encoder_mimic4.ipynb)

This notebook reproduces, at a small scale, the main experiment of

> Stefan Hegselmann, Georg von Arnim, Tillmann Rheude, Noel Kronenberg, David Sontag, Gerhard Hindricks, Roland Eils, Benjamin Wild. **Large language models are powerful electronic health record encoders.** *npj Digital Medicine* (2025). [arXiv:2502.17403](https://arxiv.org/abs/2502.17403) · [code](https://github.com/stefanhgm/ehrshot-benchmark)

**The method.** Each patient's record before the prediction time is written as Markdown (demographics, then the unique codes of each category, most recent first, as descriptions). A frozen, general-purpose LLM embedding model (Qwen3-Embedding) encodes the text, and a logistic regression is fit on the embeddings. The paper found this matches or beats count-based models and EHR-specific foundation models on EHRSHOT, without any EHR pretraining.

**In PyHealth** this is two pieces:

- `EHRTextSerializationTask` wraps any existing task (here MIMIC-IV readmission) and adds a `text` field built only from the codes that task already uses, so there is no leakage beyond the original task.
- `LLMEHREncoder` embeds the text with a frozen Hugging Face model (embeddings are cached) and puts a linear head on top. It works with PyHealth's `Trainer`, or you can fit scikit-learn's logistic regression on `model.embed(...)` as the paper does.

**What runs where.**

| Section | Data | Access | Comparable to the paper? |
|---|---|---|---|
| 1-6 | [MIMIC-IV demo](https://physionet.org/content/mimic-iv-demo/2.2/) (100 patients) | open, downloaded here | No: a smoke test. 108 samples give wide intervals. |
| 7 | MIMIC-IV | credentialed (PhysioNet) | Same protocol on a different dataset. |
| 8 | EHRSHOT | Stanford DUA | Yes: the paper's benchmark and splits. |

Reference results from the paper (EHRSHOT, macro AUROC over the 4 task groups): **Qwen3-Embedding-8B 0.761**, Qwen3-Embedding-0.6B 0.722, count-based GBM 0.719, CLMBR-T-Base 0.769.

A GPU runtime is recommended (Runtime → Change runtime type → T4). On CPU the demo still runs: embedding the records (section 4) took about 2.5 minutes in the recorded run and about 8.5 minutes in an earlier run on a busier laptop CPU.

## 0. Install

In [1]:
# On Colab, install PyHealth from the GitHub master branch, which has these models.
# Once they are in a PyPI release, `pip install pyhealth` is enough.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules or importlib.util.find_spec("pyhealth") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/sunlabuiuc/PyHealth"],
        check=True,
    )

## 1. Settings

In [2]:
import os

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

DATA = "demo"  # "demo" (open MIMIC-IV demo), "mimic4" (credentialed) or "ehrshot" (DUA)
MIMIC4_ROOT = "/content/mimic-iv/3.1"  # folder with hosp/ and icu/, for DATA="mimic4"
EHRSHOT_ROOT = "/content/ehrshot"  # for DATA="ehrshot"

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"  # the paper's best: "Qwen/Qwen3-Embedding-8B"
MAX_LENGTH = 2048  # tokens per record; the paper used up to 8192
CACHE_DIR = "./pyhealth_cache"
SEED = 0

## 2. Data: the open MIMIC-IV demo

The demo is 100 MIMIC-IV patients, published openly on PhysioNet. Only the tables this task reads are downloaded (about 700 KB).

In [3]:
import urllib.request

DEMO_URL = "https://physionet.org/files/mimic-iv-demo/2.2/"
DEMO_ROOT = "./mimic-iv-demo/2.2"
TABLES = ["hosp/patients", "hosp/admissions", "hosp/diagnoses_icd",
          "hosp/procedures_icd", "hosp/prescriptions", "icu/icustays"]

if DATA == "demo":
    for table in TABLES:
        path = os.path.join(DEMO_ROOT, table + ".csv.gz")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            urllib.request.urlretrieve(DEMO_URL + table + ".csv.gz", path)
    print("downloaded to", DEMO_ROOT)

downloaded to ./mimic-iv-demo/2.2


## 3. Serialize each sample's record as Markdown

`EHRTextSerializationTask` runs the wrapped task unchanged and writes its codes as text. `keep_inputs=True` also keeps the coded fields, for the count-based baseline in section 5.

In [4]:
from pyhealth.datasets import MIMIC4EHRDataset
from pyhealth.tasks import EHRTextSerializationTask, ReadmissionPredictionMIMIC4

if DATA in ("demo", "mimic4"):
    dataset = MIMIC4EHRDataset(
        root=DEMO_ROOT if DATA == "demo" else MIMIC4_ROOT,
        tables=["diagnoses_icd", "procedures_icd", "prescriptions"],
        cache_dir=CACHE_DIR,
    )
    base_task = ReadmissionPredictionMIMIC4()  # or MortalityPredictionMIMIC4()
    task = EHRTextSerializationTask(base_task, keep_inputs=True,
                                    max_codes_per_section=50)
    samples = dataset.set_task(task, num_workers=1)
    label_key = next(iter(base_task.output_schema))
    labels = [int(s[label_key]) for s in samples]
    print(f"{len(samples)} samples, {sum(labels)} positive ({label_key})")
    print()
    print(samples[0]["text"][:1500])

Rank 0 inferred the following `['bytes']` data format.


Rank 0 inferred the following `['no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1', 'bytes']` data format.


108 samples, 28 positive (readmission)

# Electronic Health Record

## Patient Demographics
- Gender: F
- Age: 52

## Conditions
- Portal hypertension
- Chronic airway obstruction, not elsewhere classified
- Cirrhosis of liver without mention of alcohol
- Personal history of tobacco use
- Other ascites
- Bipolar disorder, unspecified
- Posttraumatic stress disorder
- Unspecified viral hepatitis C without hepatic coma

## Procedures
- Percutaneous abdominal drainage

## Medications
- furosemide 40 MG Oral Tablet
- spironolactone 25 MG Oral Tablet
- ipratropium bromide 0.2 MG/ML Inhalation Solution
- 24 HR nicotine 0.583 MG/HR Transdermal System [Nicoderm C-Q]
- Acetaminophen 500 MG Oral Tablet [Mapap]
- emtricitabine 200 MG / tenofovir disoproxil fumarate 300 MG Oral Tablet [Truvada]
- raltegravir 400 MG Oral Tablet [Isentress]
- Potassium Chloride 10 MEQ Extended Release Oral Tablet [Klor-Con]
- furosemide 20 MG Oral Tablet
- NDA020983 60 ACTUAT albuterol 0.09 MG/ACTUAT Metered Dose In

## 4. Embed the records with a frozen LLM

The paper's protocol: patient-level train/validation/test splits, embeddings from the frozen model, then a logistic regression whose regularization is chosen on the validation set. Embeddings are cached in `CACHE_DIR`, so rerunning is fast. On a CPU the first run of the cell below takes a few minutes (about 2.5 in the recorded run, 8.5 in an earlier one).

In [5]:
import numpy as np
import torch

from pyhealth.datasets import split_by_patient
from pyhealth.models import LLMEHREncoder

# 50/20/30: on the 100-patient demo a larger test split keeps a few more positives
ratios = [0.5, 0.2, 0.3] if DATA == "demo" else [0.6, 0.2, 0.2]
train, val, test = split_by_patient(samples, ratios, seed=SEED)
rows = {name: [part[i] for i in range(len(part))]
        for name, part in [("train", train), ("val", val), ("test", test)]}
y = {name: np.array([int(r[label_key]) for r in part]) for name, part in rows.items()}
print({name: (len(v), int(v.sum())) for name, v in y.items()})

gpu = torch.cuda.is_available()
model = LLMEHREncoder(
    samples, model_name=MODEL_NAME, max_length=MAX_LENGTH, batch_size=8 if gpu else 4,
    cache_dir=os.path.join(CACHE_DIR, "embeddings"),
    torch_dtype=torch.float16 if gpu else torch.float32,
)
if gpu:
    model.to("cuda")
X = {name: model.embed([r["text"] for r in part]).cpu().numpy()
     for name, part in rows.items()}
print("embedding size", X["train"].shape[1])

{'train': (48, 14), 'val': (15, 4), 'test': (45, 10)}


embedding size 1024


## 5. Logistic regression on the embeddings, and a count-based baseline

The baseline is the paper's comparison: the same samples as bag-of-codes counts, with a logistic regression and a gradient-boosted tree model, each tuned on the validation set.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


def tuned(make, grid, X_train, y_train, X_val, y_val):
    """The model from `grid` with the best validation AUROC."""
    best = None
    for params in grid:
        clf = make(**params).fit(X_train, y_train)
        score = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        if best is None or score > best[0]:
            best = (score, clf)
    return best[1]


def counts(part):
    """Bag-of-codes counts from the task's coded fields."""
    blocks = []
    for field in base_task.input_schema:
        size = samples.input_processors[field].size()
        blocks.append(np.stack([
            np.bincount(np.asarray(r[field]).ravel().astype(int), minlength=size)[:size]
            for r in part]))
    return np.hstack(blocks).astype(float)


C_GRID = [{"C": c, "max_iter": 5000} for c in (1e-3, 1e-2, 1e-1, 1, 10, 100)]
probs = {}
llm = tuned(LogisticRegression, C_GRID, X["train"], y["train"], X["val"], y["val"])
probs["LLM embeddings + LR"] = llm.predict_proba(X["test"])[:, 1]

N = {name: counts(part) for name, part in rows.items()}
lr = tuned(LogisticRegression, C_GRID, N["train"], y["train"], N["val"], y["val"])
probs["Counts + LR"] = lr.predict_proba(N["test"])[:, 1]
GBM_GRID = [{"max_depth": d, "learning_rate": 0.05, "max_iter": 200, "random_state": SEED}
            for d in (2, 3, 4)]
gbm = tuned(HistGradientBoostingClassifier, GBM_GRID, N["train"], y["train"],
            N["val"], y["val"])
probs["Counts + GBM"] = gbm.predict_proba(N["test"])[:, 1]

## 6. Results on the test patients

AUROC with a patient-level bootstrap 95% CI, and the paired difference between the LLM encoder and each baseline. On the demo the test set has only 45 samples (10 positive), so the intervals are very wide: this checks that the pipeline runs, not the paper's claim.

In [7]:
from pyhealth.metrics import bootstrap_ci, paired_bootstrap_diff

groups = np.array([r["patient_id"] for r in rows["test"]])
print(f"test: {len(y['test'])} samples, {int(y['test'].sum())} positive\n")
for name, p in probs.items():
    ci = bootstrap_ci(y["test"], p, "roc_auc", groups=groups, n_boot=500, seed=SEED)
    print(f"{name:20s} AUROC {ci['estimate']:.3f} [{ci['lower']:.3f}, {ci['upper']:.3f}]")
print()
for name in ("Counts + LR", "Counts + GBM"):
    d = paired_bootstrap_diff(y["test"], probs["LLM embeddings + LR"], probs[name],
                              "roc_auc", groups=groups, n_boot=500, seed=SEED)
    print(f"LLM - {name:13s} {d['estimate']:+.3f} [{d['lower']:+.3f}, {d['upper']:+.3f}]")

test: 45 samples, 10 positive



LLM embeddings + LR  AUROC 0.426 [0.223, 0.726]
Counts + LR          AUROC 0.480 [0.225, 0.812]
Counts + GBM         AUROC 0.470 [0.329, 0.676]



LLM - Counts + LR   -0.054 [-0.245, +0.290]
LLM - Counts + GBM  -0.044 [-0.279, +0.259]


### The same model in a PyHealth training loop

`LLMEHREncoder` is a regular PyHealth model: the frozen encoder supplies the embeddings (from the cache above) and `Trainer` fits the linear head. Checkpoints hold only the head, not the LLM.

In [8]:
import logging

from pyhealth.datasets import get_dataloader
from pyhealth.trainer import Trainer

logging.getLogger("pyhealth.trainer").setLevel(logging.WARNING)  # quieter epochs
trainer = Trainer(model=model, metrics=["roc_auc", "pr_auc"], enable_logging=False)
trainer.train(
    train_dataloader=get_dataloader(train, batch_size=32, shuffle=True),
    val_dataloader=get_dataloader(val, batch_size=32),
    epochs=30, monitor="roc_auc", optimizer_params={"lr": 1e-2},
)
print(trainer.evaluate(get_dataloader(test, batch_size=32)))

{'roc_auc': 0.39142857142857146, 'pr_auc': 0.20038797653362345, 'loss': 0.5127914249897003}


## 7. Full MIMIC-IV (credentialed)

With PhysioNet access to [MIMIC-IV](https://physionet.org/content/mimiciv/), upload or mount the data, set `DATA = "mimic4"` and `MIMIC4_ROOT` in section 1, and rerun sections 3-6. Swap `ReadmissionPredictionMIMIC4()` for `MortalityPredictionMIMIC4()` to run mortality. Use a GPU runtime: about 400k admissions need a few GPU hours with the 0.6B model at 2048 tokens. For a quick check, run on a subset of patients first (`dataset = MIMIC4EHRDataset(..., dev=True)`).

This uses the paper's protocol on a different dataset than the paper's, so compare the LLM encoder with the count baselines here, not with the paper's absolute numbers.

## 8. EHRSHOT (the paper's benchmark)

[EHRSHOT](https://som-shahlab.github.io/ehrshot-website/) needs a Stanford data-use agreement. With the data in `EHRSHOT_ROOT`, set `DATA = "ehrshot"` and run the cell below. It wraps PyHealth's `BenchmarkEHRShot` tasks, uses EHRSHOT's own train/val/test splits, and reports AUROC per task as in the paper. The paper's macro AUROC averages 4 task groups; the 7 tasks here cover 2 of them (operational outcomes and new diagnoses; lab-value and chest X-ray tasks are not included), so the cell prints a macro over those 2 groups, which is not directly comparable with the table below.

EHRSHOT codes carry their vocabulary (`ICD10CM/E11.9`, `RxNorm/…`), which `EHRTextSerializationTask` uses to group them into sections and to look up descriptions in PyHealth's vocabularies (ICD, ATC, NDC, RxNorm). The paper's serializer also describes SNOMED, LOINC and CPT4 codes from the full OMOP vocabulary, which PyHealth does not ship; those appear here as raw codes, so expect somewhat lower scores than the paper's. Passing your own `CodeDescriber(lookup=...)` backed by an Athena vocabulary download closes that gap.

| Model (paper, EHRSHOT) | Macro AUROC |
|---|---|
| Qwen3-Embedding-8B | 0.761 |
| Qwen3-Embedding-0.6B | 0.722 |
| Count-based GBM | 0.719 |
| CLMBR-T-Base | 0.769 |

In [9]:
if DATA == "ehrshot":
    from pyhealth.datasets import EHRShotDataset
    from pyhealth.tasks import BenchmarkEHRShot

    EHRSHOT_TASKS = ["guo_los", "guo_readmission", "guo_icu", "new_hypertension",
                     "new_hyperlipidemia", "new_pancan", "new_acutemi"]
    ehrshot = EHRShotDataset(root=EHRSHOT_ROOT,
                             tables=["ehrshot", "splits", *EHRSHOT_TASKS],
                             cache_dir=CACHE_DIR)
    results = {}
    encoder = None  # loaded once; only embed() is used, so it serves every task
    for name in EHRSHOT_TASKS:
        task = EHRTextSerializationTask(BenchmarkEHRShot(task=name), demographics=None)
        task_samples = ehrshot.set_task(task)
        if encoder is None:
            encoder = LLMEHREncoder(task_samples, model_name=MODEL_NAME,
                                    max_length=MAX_LENGTH,
                                    cache_dir=os.path.join(CACHE_DIR, "embeddings"),
                                    torch_dtype=torch.float16 if gpu else torch.float32)
            if gpu:
                encoder.to("cuda")
        part = {s: [r for r in task_samples if r["split"] == s]
                for s in ("train", "val", "test")}
        Xs = {s: encoder.embed([r["text"] for r in rs]).cpu().numpy()
              for s, rs in part.items()}
        ys = {s: np.array([int(r["label"]) for r in rs]) for s, rs in part.items()}
        clf = tuned(LogisticRegression, C_GRID, Xs["train"], ys["train"],
                    Xs["val"], ys["val"])
        results[name] = roc_auc_score(ys["test"], clf.predict_proba(Xs["test"])[:, 1])
        print(f"{name:20s} AUROC {results[name]:.3f}")
    # The paper's 0.761 etc. are a macro over 4 task groups; these 7 tasks
    # cover only 2 (operational outcomes, new diagnoses), so this macro is
    # over those 2 groups and is not directly comparable with the table.
    groups = {"operational": [t for t in results if t.startswith("guo_")],
              "new diagnoses": [t for t in results if t.startswith("new_")]}
    group_means = {g: np.mean([results[t] for t in ts]) for g, ts in groups.items()}
    print("group means", {g: round(v, 3) for g, v in group_means.items()})
    print("macro AUROC over these 2 groups", np.mean(list(group_means.values())))

## Notes for the authors

- Serialization follows the paper's "unique codes, most recent first" strategy with Markdown headings. It covers diagnoses, procedures and medications as PyHealth's tasks provide them; lab values are not serialized.
- Last-token pooling with left padding, L2-normalized embeddings and a logistic regression tuned on validation follow the paper's setup for Qwen3-Embedding. Instructions are optional (`LLMEHREncoder(..., instruction="...")`).
- Feedback is welcome on GitHub issues or the PyHealth Discord.